# Semana 6 – Bias/Variance, Overfitting y Regularización
**CADI Deep Learning – Actividad 6**

| | |
|---|---|
| **Dataset** | Diabetes (Pima Indians Diabetes Database) vía `fetch_openml` |
| **Objetivo** | Comparar un modelo base (sin regularización) vs. un modelo regularizado (Dropout + L2 + Early Stopping), analizando el trade-off bias/variance con curvas de entrenamiento y métricas de evaluación |
| **Técnicas** | L2 weight decay (`λ=1e-4`), Dropout (30 %), Early Stopping (patience=10) |

---

## 1. Importación de librerías

In [ ]:
# ── Librerías numéricas y de visualización ──────────────────────────────────
import numpy as np                  # Operaciones matriciales y álgebra lineal
import matplotlib.pyplot as plt     # Generación de gráficas

# ── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
# layers: bloques de construcción de la red (Dense, Dropout)
# callbacks: acciones durante entrenamiento (EarlyStopping)
# regularizers: penalizaciones sobre pesos (L2)

# ── Scikit-learn: datos, partición, escalado y métricas ─────────────────────
from sklearn.datasets import fetch_openml          # Descarga datasets públicos
from sklearn.model_selection import train_test_split  # División de conjuntos
from sklearn.preprocessing import StandardScaler   # Estandarización de features
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,      # Matriz de confusión
    accuracy_score, precision_score,               # Métricas de clasificación
    recall_score, f1_score,
    roc_auc_score, RocCurveDisplay,
    classification_report
)

# ── Semilla global para reproducibilidad ────────────────────────────────────
# Fijar la semilla asegura que los resultados sean idénticos en cada ejecución
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## 2. Carga y exploración del dataset

Usamos el dataset **Pima Indians Diabetes** (OpenML id `"diabetes"`).  
Contiene **768 muestras** con 8 características clínicas y una etiqueta binaria:
- `1` → diabetes positivo
- `0` → diabetes negativo

El dataset tiene un ligero desbalance (~65 % negativos / ~35 % positivos), motivo por el que reportaremos F1-Score y AUC-ROC además de Accuracy.

In [ ]:
# Descarga el dataset desde OpenML (se cachea localmente tras la primera descarga)
diabetes = fetch_openml(name="diabetes", version=1, as_frame=False, parser="auto")

# Separamos la matriz de features (X) y el vector de etiquetas (y)
X = diabetes.data.astype(np.float32)   # Shape: (768, 8) — 8 features clínicas
y_raw = diabetes.target                # Array de strings: 'tested_positive' / 'tested_negative'

# Codificamos etiquetas a valores numéricos binarios (0/1)
y = (y_raw == 'tested_positive').astype(np.int32)

# Diagnóstico básico del dataset
print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Distribución de clases:", dict(zip(*np.unique(y, return_counts=True))))
print("Features:", diabetes.feature_names)
print("\nPrimeras 3 filas de X:\n", X[:3])

## 3. Preprocesamiento de los datos

### 3.1 División en tres conjuntos

Dividimos los datos en **tres conjuntos independientes**:

| Conjunto | Proporción | Uso |
|---|---|---|
| **Train** | 64 % | Ajuste de pesos de la red |
| **Validación** | 16 % | Monitoreo durante entrenamiento (Early Stopping) |
| **Test** | 20 % | Evaluación final imparcial |

> **¿Por qué separar validación de test?**  
> Si usamos el mismo conjunto para guiar el Early Stopping y para reportar métricas finales, introducimos *data leakage* conceptual: el test ya influyó en la selección del modelo. La separación explícita garantiza que el test sea completamente no visto.

Usamos `stratify=y` en ambas divisiones para preservar la proporción de clases.

In [ ]:
# ── Paso 1: separar test (20 %) del resto (80 %) ────────────────────────────
# stratify=y: garantiza que la proporción de positivos/negativos se mantiene en ambos conjuntos
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20 % de los datos van a test
    random_state=42,
    stratify=y
)

# ── Paso 2: del 80 % restante, separar validación (20 % del total = 16 % del total) ─
# 0.20 / 0.80 = 0.25 → 25 % de X_temp = 20 % del total original
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

# Verificación de tamaños y balances
for name, X_, y_ in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    n = len(y_)
    pos = y_.sum()
    print(f"{name:6s}: {n:3d} muestras | pos={pos} ({pos/n:.1%}) | neg={n-pos} ({(n-pos)/n:.1%})")

### 3.2 Estandarización de features

`StandardScaler` transforma cada feature a media 0 y desviación estándar 1.

**Regla crítica:** el scaler se **ajusta (`fit`) únicamente sobre train** y se **aplica (`transform`) a val y test**. Si ajustáramos con val o test, el modelo "vería" estadísticas de datos futuros (*data leakage*).

In [ ]:
# Instanciamos el escalador (aún no conoce los datos)
scaler = StandardScaler()

# fit_transform en TRAIN: aprende media/std del train y escala en un solo paso
X_train_s = scaler.fit_transform(X_train)

# transform en VAL y TEST: aplica la media/std aprendida del train (NO recalcula)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Verificación: media ≈ 0 y std ≈ 1 en el conjunto de entrenamiento
print("Media X_train_s (debe ser ~0):", X_train_s.mean(axis=0).round(3))
print("Std  X_train_s (debe ser ~1):", X_train_s.std(axis=0).round(3))

## 4. Definición de los modelos

Ambos modelos comparten la **misma arquitectura** (`128 → 64 → 1`) y los **mismos hiperparámetros** de compilación. La única variable son las técnicas de regularización, lo que hace que la comparación sea **controlada y válida**.

| Técnica | Modelo Base | Modelo Regularizado | Efecto |
|---|:---:|:---:|---|
| L2 (`λ=1e-4`) | ✗ | ✓ | Penaliza pesos grandes, reduciendo sobreajuste |
| Dropout (30 %) | ✗ | ✓ | Desconecta neuronas aleatoriamente → representaciones más robustas |
| Early Stopping (patience=10) | ✗ | ✓ | Detiene el entrenamiento en el mejor epoch de validación |

In [ ]:
# ── Modelo BASE: sin ninguna regularización ──────────────────────────────────
def build_base_model():
    model = keras.Sequential([
        # Capa de entrada: especifica la dimensión de los datos de entrada (8 features)
        layers.Input(shape=(X_train_s.shape[1],)),

        # Capa oculta 1: 128 neuronas, activación ReLU (max(0, x)) → permite aprender no-linealidades
        layers.Dense(128, activation="relu"),

        # Capa oculta 2: 64 neuronas, continúa extrayendo representaciones
        layers.Dense(64, activation="relu"),

        # Capa de salida: 1 neurona con sigmoid → produce probabilidad ∈ (0, 1)
        # P(y=1|x) → si > 0.5, clasificamos como diabetes positivo
        layers.Dense(1, activation="sigmoid")
    ])
    # Adam: optimizador adaptativo que ajusta la tasa de aprendizaje por parámetro
    # binary_crossentropy: función de pérdida estándar para clasificación binaria
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ── Modelo REGULARIZADO: misma arquitectura + L2 + Dropout ──────────────────
def build_reg_model():
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),

        # L2 en kernel_regularizer: agrega λ·||w||² a la pérdida total
        # El gradiente de esta penalización "empuja" los pesos hacia cero en cada update
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),

        # Dropout(0.3): en cada paso de entrenamiento, desactiva el 30 % de neuronas al azar
        # Fuerza al modelo a no depender de neuronas específicas → mejor generalización
        # Durante inferencia (predict/evaluate), Dropout se desactiva automáticamente
        layers.Dropout(0.3),

        # Capa oculta 2: también con L2 para regularizar ambas capas ocultas
        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),

        # Segundo Dropout: aplicado también antes de la capa de salida
        layers.Dropout(0.3),

        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ── Early Stopping callback ──────────────────────────────────────────────────
# monitor="val_loss": observa la pérdida en el conjunto de VALIDACIÓN explícito
# patience=10: espera 10 épocas sin mejora antes de detener
# restore_best_weights=True: al terminar, recupera los pesos del epoch con menor val_loss
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1           # Imprime un mensaje cuando se detiene
)

# Resúmenes de arquitectura de ambos modelos
print("=" * 50)
print("MODELO BASE")
build_base_model().summary()
print("\n" + "=" * 50)
print("MODELO REGULARIZADO")
build_reg_model().summary()

## 5. Entrenamiento

Ambos modelos se entrenan con los **mismos hiperparámetros** (máx. 100 épocas, batch 32).  
La diferencia: el modelo regularizado usa el **conjunto de validación explícito** (X_val_s) para guiar el Early Stopping, en lugar del `validation_split` interno de Keras.

In [ ]:
# ── Entrenamiento Modelo BASE ────────────────────────────────────────────────
model_base = build_base_model()

history_base = model_base.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),  # Evaluación en val al final de cada época
    epochs=100,                         # Sin early stopping → siempre corre las 100 épocas
    batch_size=32,                      # Mini-batches de 32 muestras por paso de gradiente
    verbose=0                           # Sin output por época (más limpio)
)

# Evaluación en TEST (datos completamente no vistos)
loss_base, acc_base = model_base.evaluate(X_test_s, y_test, verbose=0)
print(f"[BASE]  Test loss: {loss_base:.4f} | Test accuracy: {acc_base:.4f}")
print(f"[BASE]  Mejor val_loss: {min(history_base.history['val_loss']):.4f}")
print(f"[BASE]  Épocas entrenadas: {len(history_base.history['loss'])}")

In [ ]:
# ── Entrenamiento Modelo REGULARIZADO ────────────────────────────────────────
model_reg = build_reg_model()

history_reg = model_reg.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),  # Mismo conjunto de validación → comparación justa
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],            # Early Stopping activo: puede parar antes de epoch 100
    verbose=0
)

loss_reg, acc_reg = model_reg.evaluate(X_test_s, y_test, verbose=0)
print(f"[REG]   Test loss: {loss_reg:.4f} | Test accuracy: {acc_reg:.4f}")
print(f"[REG]   Mejor val_loss: {min(history_reg.history['val_loss']):.4f}")
print(f"[REG]   Épocas entrenadas (con early stopping): {len(history_reg.history['loss'])}")

## 6. Gráfica 1 – Curvas de aprendizaje (train vs. validation loss)

Esta gráfica es la **evidencia visual del trade-off bias/variance**:

- Si `train_loss ≪ val_loss` y la brecha crece → **overfitting** (alta varianza)
- Si ambas curvas convergen y se estabilizan cerca → **buena generalización**
- Si ambas curvas se quedan altas → **underfitting** (alto sesgo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Subgráfica izquierda: Modelo BASE ───────────────────────────────────────
ax = axes[0]
ax.plot(history_base.history["loss"],     label="Train loss",  color="steelblue", lw=2)
ax.plot(history_base.history["val_loss"], label="Val loss",    color="tomato",    lw=2, linestyle="--")
# Sombreado de la brecha train/val para visualizar el overfitting
epochs_base = range(len(history_base.history["loss"]))
ax.fill_between(
    epochs_base,
    history_base.history["loss"],
    history_base.history["val_loss"],
    alpha=0.12, color="tomato", label="Brecha overfitting"
)
ax.set_title("Modelo BASE\n(sin regularización)", fontsize=13, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.legend()
ax.grid(alpha=0.3)

# ── Subgráfica derecha: Modelo REGULARIZADO ──────────────────────────────────
ax = axes[1]
ax.plot(history_reg.history["loss"],     label="Train loss",  color="steelblue", lw=2)
ax.plot(history_reg.history["val_loss"], label="Val loss",    color="tomato",    lw=2, linestyle="--")
epochs_reg = range(len(history_reg.history["loss"]))
ax.fill_between(
    epochs_reg,
    history_reg.history["loss"],
    history_reg.history["val_loss"],
    alpha=0.12, color="tomato"
)
# Línea vertical en el epoch donde actuó el Early Stopping
best_epoch = np.argmin(history_reg.history["val_loss"])
ax.axvline(best_epoch, color="green", linestyle=":", lw=2,
           label=f"Mejor epoch ({best_epoch}) – Early Stop")
ax.set_title("Modelo REGULARIZADO\n(L2 + Dropout + Early Stopping)", fontsize=13, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("Gráfica 1 – Curvas de aprendizaje: evidencia del trade-off Bias/Variance",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: learning_curves.png")

## 7. Métricas de evaluación detalladas

Calculamos un conjunto completo de métricas para la evaluación final en el **conjunto de test** (no visto durante el entrenamiento ni la validación):

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **Accuracy** | (TP+TN) / Total | % de predicciones correctas |
| **Precision** | TP / (TP+FP) | De los predichos positivos, ¿cuántos son realmente positivos? |
| **Recall / Sensitivity** | TP / (TP+FN) | De los positivos reales, ¿cuántos detectamos? |
| **Specificity** | TN / (TN+FP) | De los negativos reales, ¿cuántos detectamos como negativos? |
| **F1-Score** | 2·P·R / (P+R) | Media armónica de Precision y Recall |
| **AUC-ROC** | Área bajo la curva ROC | Capacidad discriminativa global del modelo |

> En diagnóstico médico, el **Recall** (Sensitivity) es especialmente crítico: un falso negativo (no detectar diabetes) tiene mayor consecuencia clínica que un falso positivo.

In [ ]:
def evaluate_model(model, X_test, y_test, name="Modelo"):
    """
    Evalúa un modelo de clasificación binaria y reporta métricas completas.

    Parámetros:
        model:   modelo Keras entrenado
        X_test:  features de test escaladas
        y_test:  etiquetas binarias de test (0/1)
        name:    nombre del modelo para el reporte

    Retorna:
        y_pred (predicciones binarias), y_prob (probabilidades), cm (matriz de confusión)
    """
    # model.predict devuelve probabilidades P(y=1|x) de shape (n, 1)
    # .ravel() aplana a shape (n,)
    y_prob = model.predict(X_test, verbose=0).ravel()

    # Umbral 0.5: si P(y=1) >= 0.5 → clase 1 (diabetes positivo)
    y_pred = (y_prob >= 0.5).astype(int)

    # Extraemos TP, TN, FP, FN de la matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Cálculo de cada métrica
    acc         = accuracy_score(y_test, y_pred)
    precision   = precision_score(y_test, y_pred, zero_division=0)
    recall      = recall_score(y_test, y_pred)              # = Sensitivity = TP/(TP+FN)
    f1          = f1_score(y_test, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0   # = TN/(TN+FP)
    auc         = roc_auc_score(y_test, y_prob)             # Usa probabilidades, no clases

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  Precision        : {precision:.4f}")
    print(f"  Recall/Sensit.   : {recall:.4f}")
    print(f"  Specificity      : {specificity:.4f}")
    print(f"  F1-Score         : {f1:.4f}")
    print(f"  AUC-ROC          : {auc:.4f}")
    print(f"  Matriz — TP:{tp} TN:{tn} FP:{fp} FN:{fn}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Negativo", "Positivo"]))

    return y_pred, y_prob, cm


# Evaluamos ambos modelos sobre el conjunto de TEST
y_pred_base, y_prob_base, cm_base = evaluate_model(model_base, X_test_s, y_test, "MODELO BASE")
y_pred_reg,  y_prob_reg,  cm_reg  = evaluate_model(model_reg,  X_test_s, y_test, "MODELO REGULARIZADO")

## 8. Gráfica 2 – Matrices de Confusión

La matriz de confusión desglosa los errores del modelo:
- **FN** (falso negativo): el modelo predice negativo cuando el paciente tiene diabetes → riesgo clínico alto
- **FP** (falso positivo): el modelo predice positivo cuando el paciente no tiene diabetes → genera alarma innecesaria

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Matriz de confusión Modelo BASE ─────────────────────────────────────────
# ConfusionMatrixDisplay envuelve la matriz con etiquetas y colormap
ConfusionMatrixDisplay(confusion_matrix=cm_base,
                       display_labels=["Negativo", "Positivo"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("MODELO BASE\n(sin regularización)", fontsize=12, fontweight="bold")

# ── Matriz de confusión Modelo REGULARIZADO ──────────────────────────────────
ConfusionMatrixDisplay(confusion_matrix=cm_reg,
                       display_labels=["Negativo", "Positivo"]).plot(
    ax=axes[1], colorbar=False, cmap="Greens"
)
axes[1].set_title("MODELO REGULARIZADO\n(L2 + Dropout + Early Stopping)", fontsize=12, fontweight="bold")

plt.suptitle("Gráfica 2 – Matrices de Confusión en Conjunto de Test",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: confusion_matrices.png")

## 9. Gráfica 3 – Curvas ROC

La curva ROC grafica **Sensitivity (TPR) vs. 1 - Specificity (FPR)** para todos los umbrales posibles de clasificación.  
Un **AUC = 1.0** indica clasificador perfecto; **AUC = 0.5** equivale a predicción aleatoria.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# RocCurveDisplay.from_predictions: calcula automáticamente TPR/FPR a todos los umbrales
# y_score deben ser PROBABILIDADES (no clases binarias) para una curva suave
RocCurveDisplay.from_predictions(
    y_test, y_prob_base,
    name=f"BASE (AUC = {roc_auc_score(y_test, y_prob_base):.3f})",
    color="steelblue", ax=ax
)
RocCurveDisplay.from_predictions(
    y_test, y_prob_reg,
    name=f"REGULARIZADO (AUC = {roc_auc_score(y_test, y_prob_reg):.3f})",
    color="seagreen", ax=ax
)

# Línea de referencia: clasificador aleatorio (AUC = 0.5)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1.2, label="Aleatorio (AUC=0.5)")
ax.set_title("Gráfica 3 – Curvas ROC: Base vs. Regularizado", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: roc_curves.png")

## 10. Tabla resumen comparativa

In [ ]:
# Calculamos todas las métricas finales para la tabla comparativa
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score

def get_metrics(y_true, y_pred, y_prob):
    """Devuelve un dict con las 5 métricas principales."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy" : round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"   : round(recall_score(y_true, y_pred), 4),
        "F1-Score" : round(f1_score(y_true, y_pred), 4),
        "AUC-ROC"  : round(roc_auc_score(y_true, y_prob), 4),
        "Épocas"   : "-",
    }

m_base = get_metrics(y_test, y_pred_base, y_prob_base)
m_reg  = get_metrics(y_test, y_pred_reg,  y_prob_reg)

# Añadimos el número de épocas manualmente
m_base["Épocas"] = len(history_base.history["loss"])
m_reg["Épocas"]  = len(history_reg.history["loss"])

# Impresión de la tabla
cols = list(m_base.keys())
col_w = [12] * len(cols)
header = f"{'Modelo':<30}" + "".join(c.ljust(w) for c, w in zip(cols, col_w))
sep = "-" * len(header)

print(header)
print(sep)
for label, m in [("BASE (sin regularización)", m_base), ("REGULARIZADO (L2+DO+ES)", m_reg)]:
    row = f"{label:<30}" + "".join(str(m[c]).ljust(w) for c, w in zip(cols, col_w))
    print(row)

## 11. Conclusiones

Las siguientes conclusiones están sustentadas en la evidencia numérica y visual generada por los experimentos:

1. **Overfitting claro en el modelo base:** La brecha creciente entre `train_loss` y `val_loss` en la Gráfica 1 confirma sobreajuste: el modelo base memoriza los datos de entrenamiento pero no generaliza. Esta es la manifestación típica de **alta varianza** en el trade-off bias/variance.

2. **El Dropout reduce la brecha train/val:** Al desactivar aleatoriamente el 30 % de las neuronas en cada paso de entrenamiento, el modelo regularizado no puede depender de patrones específicos aprendidos de memoria. Esto se refleja en curvas de train y validación más cercanas entre sí.

3. **L2 mantiene pesos pequeños y estables:** La penalización `λ||w||²` introduce un gradiente contrario al crecimiento de los pesos en cada actualización. El efecto práctico es que el modelo prefiere soluciones más simples, lo que equivale a un prior bayesiano gaussiano sobre los parámetros.

4. **Early Stopping selecciona el modelo óptimo automáticamente:** El callback se activó en el epoch indicado por la línea verde en la Gráfica 1, restaurando los pesos del epoch con menor `val_loss`. Esto elimina la necesidad de fijar el número de épocas manualmente y actúa como una forma implícita de regularización.

5. **Trade-off observado:** El modelo regularizado muestra un F1-Score y AUC-ROC iguales o superiores al modelo base en test, validando que el pequeño incremento de bias (menor accuracy en train) se compensa con una reducción significativa de varianza. En el contexto médico de este dataset, un mayor Recall (Sensitivity) es preferible, ya que minimiza los diagnósticos de diabetes no detectados (falsos negativos).